# ☀️ Solar Filament Segmentation 2026: EdgeAttNet Pipeline

Welcome to this advanced baseline notebook for the **Solar Filament Segmentation Challenge 2026**[cite: 1]. 

The objective of this competition is to perform pixel-perfect segmentation of solar filaments from 2048x2048 H-Alpha observations[cite: 1]. Filaments are dense clouds of cool plasma suspended above the solar surface by complex magnetic fields[cite: 1]. Accurately segmenting them is critical for space weather forecasting, as the destabilization of these structures is linked to Coronal Mass Ejections (CMEs) that can severely disrupt Earth's power grids and satellite networks[cite: 1].

### 🎯 The Challenge: Capturing "Barbs"
While the main body of a filament (the spine) is relatively easy to detect, the true challenge lies in segmenting the incredibly fine, thread-like lateral protrusions known as **barbs**[cite: 1]. Identifying the orientation of these barbs dictates the magnetic chirality (handedness) of the filament, which directly correlates to the severity of a resulting solar storm[cite: 1]. 

Standard architectures like U-Net often smooth over these critical barbs, and instance segmentation models can sometimes fragment the filament into disconnected "islands"[cite: 1]. 

### 🚀 Our Approach: EdgeAttNet
To overcome this, this notebook implements **EdgeAttNet**, a state-of-the-art architecture explicitly engineered for the MAGFILO dataset[cite: 1]. EdgeAttNet utilizes an Edge-Guided Multi-Head Self-Attention (EG-MHSA) module at the deepest network bottleneck[cite: 1]. 

By extracting a learnable edge map directly from the input image and applying linear transformations to the Query ($Q$) and Key ($K$) matrices, the network integrates a structural edge prior directly into the self-attention formula[cite: 1]:

$$Attention(Q,K,V)=softmax(\frac{QK^{\top}}{\sqrt{d_{k}}})V$$

This explicitly forces the attention heads to calculate weights heavily biased toward the filament boundaries and barbs, entirely eliminating the need for bulky positional encodings while maintaining lightning-fast inference[cite: 1].

### 📋 Optimization Updates
To maximize the Panoptic Quality (PQ) metric and prevent score collapse[cite: 1], this updated pipeline features:
1. **High-Resolution Augmentations:** Training at 1024x1024 resolution with random flips to preserve barb topology.
2. **Gradient Accumulation:** Simulating larger batch sizes to fit inside Kaggle's GPU memory limits safely.
3. **Morphological Closing:** Post-processing predictions with OpenCV to connect fragmented "islands" before generating the strict COCO Run-Length Encoding (RLE) submission file[cite: 1, 2].

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install pycocotools torchvision

import os
import cv2
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from pycocotools import mask as mask_util
import torchvision.transforms.functional as TF

# Ensure deterministic behavior where possible
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 1. Data Parsing and High-Resolution Augmentation

The MAGFILO 1.0 dataset provides strictly 2048x2048 grayscale observations[cite: 1]. To preserve the fine morphological structures (barbs) while managing Kaggle's GPU memory, we will load the images at 1024x1024 resolution and apply random horizontal and vertical flips during training.

In [ ]:
class MAGFILOEdgeDataset(Dataset):
    """
    Custom Dataset for the MAGFILO 1.0 Ecosystem.
    Parses the COCO JSON schema, applies augmentations, and generates mask targets.
    """
    def __init__(self, root_dir, annotation_file, target_size=(1024, 1024)):
        self.root_dir = root_dir
        self.coco = COCO(annotation_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.target_size = target_size

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        # Retrieve image info and path
        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        coco_anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs([img_id])[0]
        
        img_path = os.path.join(self.root_dir, img_info['file_name'])
        
        # Load grayscale image and keep it as a single channel 
        image = Image.open(img_path).convert("L")
        
        # Generate aggregate semantic mask for all instances
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in coco_anns:
            instance_mask = self.coco.annToMask(ann)
            mask = np.maximum(mask, instance_mask) 
            
        mask = Image.fromarray(mask)

        # Scale target size to preserve fine structures
        if self.target_size:
            image = TF.resize(image, self.target_size, interpolation=Image.BILINEAR)
            mask = TF.resize(mask, self.target_size, interpolation=Image.NEAREST)

        # Data Augmentations
        if random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if random.random() > 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)

        # Convert to Tensors
        image_tensor = TF.to_tensor(image)
        mask_tensor = torch.as_tensor(np.array(mask), dtype=torch.float32).unsqueeze(0)

        return image_tensor, mask_tensor

## 2. Defining the EdgeAttNet Architecture

We construct the `EdgeAttNet` using a symmetric U-Net backbone up to 512 channels[cite: 1]. The structural innovation is the `EdgeGuidedMHSA` layer at the bottleneck[cite: 1]. 

*Fix included:* The edge map interpolation dynamically scales to `size=(bottleneck.shape[2], bottleneck.shape[3])` to prevent spatial mismatch crashes.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class EdgeGuidedMHSA(nn.Module):
    def __init__(self, channels=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads

        self.q_proj = nn.Linear(channels, channels)
        self.k_proj = nn.Linear(channels, channels)
        self.v_proj = nn.Linear(channels, channels)
        
        self.edge_to_q = nn.Linear(1, channels)
        self.edge_to_k = nn.Linear(1, channels)
        
        self.out_proj = nn.Linear(channels, channels)
        self.norm = nn.LayerNorm(channels)

    def forward(self, x, edge_map):
        B, C, H, W = x.shape
        N = H * W
        
        x_flat = x.view(B, C, N).permute(0, 2, 1)
        edge_flat = edge_map.view(B, 1, N).permute(0, 2, 1)

        q = self.q_proj(x_flat) * self.edge_to_q(edge_flat)
        k = self.k_proj(x_flat) * self.edge_to_k(edge_flat)
        v = self.v_proj(x_flat)

        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        attn = F.scaled_dot_product_attention(q, k, v)
        attn = attn.transpose(1, 2).reshape(B, N, C)
        
        out = self.out_proj(attn)
        out = self.norm(out + x_flat)
        return out.permute(0, 2, 1).view(B, C, H, W)

import torch
import torch.nn as nn
import torch.nn.functional as F

class EdgeAttNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.edge_extractor = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )
        
        # 4-stage encoder: Reduces spatial resolution by a factor of 16
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 512)
        
        self.eg_mhsa1 = EdgeGuidedMHSA(channels=512, num_heads=4)
        self.eg_mhsa2 = EdgeGuidedMHSA(channels=512, num_heads=4)
        
        # Decoder 1: Upsample 512 -> Concat with x4 (512) = 1024 -> Reduce to 256
        self.up1 = nn.ConvTranspose2d(512, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 256)
        
        # Decoder 2: Upsample 256 -> Concat with x3 (256) = 512 -> Reduce to 128
        self.up2 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 128)
        
        # Decoder 3: Upsample 128 -> Concat with x2 (128) = 256 -> Reduce to 64
        self.up3 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 64)

        # Decoder 4: Upsample 64 -> Concat with x1 (64) = 128 -> Maintain at 64
        self.up4 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)
        
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        edge_map = self.edge_extractor(x)

        x1 = self.down1(x)
        x2 = self.down2(self.pool(x1))
        x3 = self.down3(self.pool(x2))
        x4 = self.down4(self.pool(x3))
        
        b = self.bottleneck(self.pool(x4))
        
        edge_map_b = F.interpolate(edge_map, size=(b.shape[2], b.shape[3]), mode='bilinear', align_corners=False)
        b = self.eg_mhsa1(b, edge_map_b)
        b = self.eg_mhsa2(b, edge_map_b)
        
        u1 = self.conv_up1(torch.cat([self.up1(b), x4], dim=1))
        u2 = self.conv_up2(torch.cat([self.up2(u1), x3], dim=1))
        u3 = self.conv_up3(torch.cat([self.up3(u2), x2], dim=1))
        u4 = self.conv_up4(torch.cat([self.up4(u3), x1], dim=1))
        
        return self.final_conv(u4)

## 3. Topologically Safe Training & Loss Functions

To optimize for the competition's Panoptic Quality and Mean Dice scoring metrics, we employ a composite BCE + Dice loss function[cite: 1]. This heavily penalizes false positives and structural fragmentation.

In [ ]:
import torch
import torch.nn.functional as F

def dice_loss(pred_probs, target, smooth=1e-6):
    pred_probs = pred_probs.contiguous()
    target = target.contiguous()    
    intersection = (pred_probs * target).sum(dim=2).sum(dim=2)
    loss = (1 - ((2. * intersection + smooth) / (pred_probs.sum(dim=2).sum(dim=2) + target.sum(dim=2).sum(dim=2) + smooth)))
    return loss.mean()

def bce_dice_loss(pred_logits, target):
    # Safe FP16 calculation for BCE
    bce = F.binary_cross_entropy_with_logits(pred_logits, target)
    
    # Manually apply sigmoid to get probabilities for the Dice Loss
    pred_probs = torch.sigmoid(pred_logits)
    dice = dice_loss(pred_probs, target)
    
    return bce + dice

In [ ]:
import torch
from torch.utils.data import random_split, DataLoader
from torchmetrics.segmentation import DiceScore

# 1. Define the hardware device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Define the dataset directory paths
TRAIN_IMG_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images"
TRAIN_ANN_FILE = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

# 3. Initialize dataset
full_dataset = MAGFILOEdgeDataset(root_dir=TRAIN_IMG_DIR, annotation_file=TRAIN_ANN_FILE)

# 4. 80/20 Topological Safe Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# 5. Initialize the metric
dice_metric = DiceScore(num_classes=1).to(device)

# 6. Initialize the model before calling .eval()
model = EdgeAttNet().to(device)

# 7. Validation Loop
model.eval()
val_dice = 0.0

with torch.no_grad():
    for val_images, val_masks in val_loader:
        val_images, val_masks = val_images.to(device), val_masks.to(device)
        preds = model(val_images)
        pred_probs = torch.sigmoid(preds)
        
        # Binarize masks for exact Dice calculation
        pred_binary = (pred_probs > 0.5).int()
        val_dice += dice_metric(pred_binary, val_masks.int()).item()

avg_val_dice = val_dice / len(val_loader)
print(f"Epoch Local Validation Dice: {avg_val_dice:.4f}")

## 4. Gradient Accumulation Training Loop

Because we are processing dense 1024x1024 arrays, we use a batch size of 1 to prevent OOM errors and `ACCUMULATION_STEPS = 4` to stabilize the gradients safely.

In [ ]:
import gc
from torch.amp import autocast, GradScaler # Updated AMP import syntax

TRAIN_IMG_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images"
TRAIN_ANN_FILE = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print("Initializing MAGFILO Training Dataset at 1024x1024...")
train_dataset = MAGFILOEdgeDataset(
    root_dir=TRAIN_IMG_DIR, 
    annotation_file=TRAIN_ANN_FILE, 
    target_size=(1024, 1024) 
)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EdgeAttNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)

# --- Updated Scaler Initialization ---
scaler = GradScaler('cuda')

NUM_EPOCHS = 15 
ACCUMULATION_STEPS = 4 

print(f"Starting AMP Training for {NUM_EPOCHS} epochs on {device}...")

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    optimizer.zero_grad()
    
    for i, (images, masks) in enumerate(train_loader):
        images, masks = images.to(device), masks.to(device)
        
        # --- Updated Autocast Syntax ---
        with autocast('cuda'):
            preds = model(images)
            loss = bce_dice_loss(preds, masks)
            loss = loss / ACCUMULATION_STEPS
            
        scaler.scale(loss).backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        epoch_loss += (loss.item() * ACCUMULATION_STEPS)
        
        del images, masks, preds
        torch.cuda.empty_cache()
        gc.collect()
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Avg BCE+Dice Loss: {avg_loss:.4f}")

print("Training Complete! Weights are optimized for filament morphology.")

## 5. Inference and Generating the Submission

The Panoptic Quality metric rapidly collapses if an algorithm generates dozens of disconnected masks representing a single physical filament[cite: 1]. To combat this, we apply **Morphological Closing** (`cv2.MORPH_CLOSE`) to solidify the predicted filament spines before separating the instances.

Additionally, we ensure the predicted arrays are resized back to the exact 2048x2048 space and converted to Fortran-contiguous arrays before RLE encoding, strictly meeting the Kaggle submission requirements[cite: 1, 2].

In [ ]:
def generate_rle_submission_optimized(model, test_dir, device, threshold=0.5):
    """
    Runs inference with Morphological Closing to prevent PQ score collapse
    from over-fragmentation, outputting to a compliant submission CSV.
    """
    model.eval()
    submission_data = []
    
    # 1. Safety Check to prevent silent halts
    if not os.path.exists(test_dir):
        print(f"CRITICAL ERROR: Test directory not found at {test_dir}")
        print("Saving empty submission.csv to prevent Kaggle backend failure.")
        df = pd.DataFrame(columns=["filament_id", "segmentation_rle"])
        df.to_csv("submission.csv", index=False)
        return
        
    test_files = [f for f in os.listdir(test_dir) if f.endswith(('.jpeg', '.jpg'))]
    print(f"Found {len(test_files)} test images. Starting optimized inference...")
    
    # Define the morphological kernel (5x5 ellipse)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    
    for filename in test_files:
        base_id = os.path.splitext(filename)[0]
        img_path = os.path.join(test_dir, filename)
        
        image = Image.open(img_path).convert("L")
        orig_w, orig_h = image.size
        
        image_resized = TF.resize(image, (1024, 1024), interpolation=Image.BILINEAR)
        img_tensor = TF.to_tensor(image_resized).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred_logits = model(img_tensor)
        
        # Apply Sigmoid to convert logits to probabilities
        pred_probs = torch.sigmoid(pred_logits)
        
        # Resize prediction back to 2048x2048 native space
        pred_resized = F.interpolate(pred_probs, size=(orig_h, orig_w), mode='bilinear', align_corners=False)
        pred_mask = (pred_resized.squeeze().cpu().numpy() > threshold).astype(np.uint8)
        
        # Morphological Closing to bridge fragmented "islands"
        pred_mask_closed = cv2.morphologyEx(pred_mask, cv2.MORPH_CLOSE, kernel)
        
        # Post-processing: Label connected components as individual instances
        num_labels, labels = cv2.connectedComponents(pred_mask_closed)
        
        # Handle the empty mask edge-case logically
        if num_labels == 1:
            empty_mask = np.asfortranarray(np.zeros((orig_h, orig_w), dtype=np.uint8))
            encoded = mask_util.encode(empty_mask)
            submission_data.append({
                "filament_id": f"{base_id}_1",
                "segmentation_rle": encoded['counts'].decode('utf-8')
            })
            continue

        # Define a minimum pixel area (e.g., 100 pixels) to ignore noise
        MIN_AREA = 100 

        for i in range(1, num_labels):
            instance_mask = (labels == i).astype(np.uint8)
            
            # Filter out tiny artifacts that ruin Panoptic Quality
            if instance_mask.sum() < MIN_AREA:
                continue
                
            # CRUCIAL: Convert to Fortran-contiguous array for pycocotools
            fortran_mask = np.asfortranarray(instance_mask)
            encoded = mask_util.encode(fortran_mask)
            
            submission_data.append({
                "filament_id": f"{base_id}_{i}",
                "segmentation_rle": encoded['counts'].decode('utf-8')
            })
            
    # Explicitly define columns to match Kaggle's required schema
    df = pd.DataFrame(submission_data, columns=["filament_id", "segmentation_rle"])
    df.to_csv("submission.csv", index=False)
    print(f"Success! submission.csv saved with {len(df)} contiguous masks.")

# --- EXECUTE INFERENCE ---

TEST_IMG_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"

generate_rle_submission_optimized(model, TEST_IMG_DIR, device, threshold=0.5)